# Information Retrieval System

In [11]:
from datasets import load_dataset
import nltk
from rank_bm25 import BM25Okapi
from tqdm import tqdm
import numpy as np

In [ ]:
# Load Dataset
# Load corpus (100k subset)
corpus_dataset = load_dataset(
    "ms_marco",
    "v2.1",
    split="train[:100000]"
)

# Load validation queries
queries_dataset = load_dataset(
    "ms_marco",
    "v2.1",
    split="validation[:1000]"
)

In [ ]:
print(corpus_dataset)

Dataset({
    features: ['answers', 'passages', 'query', 'query_id', 'query_type', 'wellFormedAnswers'],
    num_rows: 100000
})


In [ ]:
# Extract Passages
corpus_passages = []

for doc in corpus_dataset["passages"]:
    corpus_passages.extend(doc["passage_text"])

In [ ]:
print("Total passages:", len(corpus_passages))
print("Total queries:", len(queries_dataset))

Total passages: 997459
Total queries: 1000


## BM25 Retriever

In [ ]:
# Create Tokenizer Function
def tokenize(text):
    return nltk.word_tokenize(text.lower())

In [ ]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to C:\Users\PAVAN
[nltk_data]     TEJA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\PAVAN
[nltk_data]     TEJA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# LT
# TokenizeEntire Corpus
tokenized_corpus = [
    tokenize(doc)
    for doc in tqdm(corpus_passages)
]

In [ ]:
# LT
import pickle

with open("tokenized_corpus.pkl", "wb") as f:
    pickle.dump(tokenized_corpus, f)

In [ ]:
import pickle
with open("tokenized_corpus.pkl", "rb") as f:
    tokenized_corpus = pickle.load(f)

In [ ]:
# Build BM25 Index
bm25 = BM25Okapi(tokenized_corpus)

In [ ]:
# Create BM25 search function
def bm25_search(query_text, top_k=100):
    tokenized_query = tokenize(query_text)

    scores = bm25.get_scores(tokenized_query)

    top_indices = scores.argsort()[-top_k:][::-1]

    return top_indices, scores[top_indices]

In [ ]:

# Test BM25
query = "what is artificial intelligence"

indices, scores = bm25_search(query, top_k=5)

for rank, (idx, score) in enumerate(zip(indices, scores), start=1):
    print(f"Rank {rank}")
    print(f"Document Index: {idx}")
    print(f"Score: {score}")
    print(corpus_passages[idx])
    print("-" * 80)

In [ ]:
print("BM25 Index Built Successfully!")
print(f"Corpus Size: {len(corpus_passages)} passages")

BM25 Index Built Successfully!
Corpus Size: 997459 passages


## Dense Vector Search using FAISS

In [ ]:
import faiss
import numpy as np
import os
from sentence_transformers import SentenceTransformer

In [ ]:
os.makedirs("output",exist_ok=True)

In [ ]:
# Load Sentence Transformer Model
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# LT
# Generate Corpus Embeddings
corpus_embeddings = model.encode(
    corpus_passages,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True
)

In [ ]:
# Save Corpus Embeddings
np.save(
    "output/corpus_embeddings.npy",
    corpus_embeddings
) 

In [ ]:
corpus_embeddings = np.load("output/corpus_embeddings.npy")

In [ ]:
# Normalize Embeddings
faiss.normalize_L2(corpus_embeddings)

In [ ]:
# Get Embedding Dimension
dimension = corpus_embeddings.shape[1]
print(dimension)

384


In [ ]:
# Creating FAISS Index
dense_index = faiss.IndexFlatIP(dimension)

In [ ]:
# Adding Embeddings to FAISS
dense_index.add(corpus_embeddings)

In [ ]:
# Save FAISS Index
faiss.write_index(
    dense_index,
    "output/dense.index"
)

In [ ]:
# Dense Search Function
def dense_search(query_text, top_k=100):

    query_embedding = model.encode(
        [query_text],
        convert_to_numpy=True
    )

    faiss.normalize_L2(query_embedding)

    distances, indices = dense_index.search(
        query_embedding,
        top_k
    )

    return indices[0], distances[0]

In [ ]:
# Test Dense Search
query = "what is artificial intelligence"

indices, scores = dense_search(
    query,
    top_k=5
)

for rank, (idx, score) in enumerate(
    zip(indices, scores),
    start=1
):
    print(f"Rank {rank}")
    print(f"Document Index: {idx}")
    print(f"Score: {score}")
    print(corpus_passages[idx])
    print("-" * 80)

Rank 1
Document Index: 244573
Score: 0.8726219534873962
Artificial Intelligence (AI) Artificial intelligence (AI) is an area of computer science that emphasizes the creation of intelligent machines that work and react like humans. Some of the activities computers with artificial intelligence are designed for include: 1  Speech recognition.  Learning.
--------------------------------------------------------------------------------
Rank 2
Document Index: 407007
Score: 0.8691966533660889
Artificial intelligence (AI) is an area of computer science that emphasizes the creation of intelligent machines that work and react like humans. Some of the activities computers with artificial intelligence are designed for include: 1  Speech recognition.  Learning.
--------------------------------------------------------------------------------
Rank 3
Document Index: 407001
Score: 0.8580491542816162
Artificial intelligence (AI) is an area of computer science that emphasizes the creation of intelligent m

In [ ]:
print(os.path.exists("output/corpus_embeddings.npy"))
print(os.path.exists("output/dense.index"))

True
True


## Hybrid Retrieval using RRF

In [ ]:
import time
import json
import pickle
import os
import numpy as np
import pytrec_eval

In [ ]:
# Save BM25 Index

with open("output/bm25_index.pkl","wb") as f:
    pickle.dump(bm25,f)

In [ ]:
# Hybrid Search Function (RFF)
def hybrid_search(query_text, top_k=100, k=60):

    bm25_indices, _ = bm25_search(
        query_text,
        top_k=top_k
    )

    dense_indices, _ = dense_search(
        query_text,
        top_k=top_k
    )

    rrf_scores = {}

    for rank, doc_id in enumerate(bm25_indices):

        if doc_id not in rrf_scores:
            rrf_scores[doc_id] = 0

        rrf_scores[doc_id] += 1 / (k + rank + 1)

    for rank, doc_id in enumerate(dense_indices):

        if doc_id not in rrf_scores:
            rrf_scores[doc_id] = 0

        rrf_scores[doc_id] += 1 / (k + rank + 1)

    sorted_results = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    return [doc_id for doc_id, _ in sorted_results[:top_k]]

In [ ]:
# Test Hybrid Search
query = "what is artificial intelligence"

results = hybrid_search(
    query,
    top_k=5
)

for rank, idx in enumerate(results, start=1):

    print(f"Rank {rank}")
    print(f"Document Index: {idx}")
    print(corpus_passages[idx])
    print("-"*80)

Rank 1
Document Index: 244573
Artificial Intelligence (AI) Artificial intelligence (AI) is an area of computer science that emphasizes the creation of intelligent machines that work and react like humans. Some of the activities computers with artificial intelligence are designed for include: 1  Speech recognition.  Learning.
--------------------------------------------------------------------------------
Rank 2
Document Index: 578822
Artificial intelligence is technology that is designed to learn and self-improve. It is typically used to solve complex problems that are impossible to tackle with traditional code. In some cases, artificial intelligence research and development programs aim to replicate aspects of human intelligence or alternate types of intelligence that may exceed human abilities in certain respects. The following are common types of artificial intelligence.
--------------------------------------------------------------------------------
Rank 3
Document Index: 948577
In

## BenchMarking and Evaluation

In [ ]:
# BM25 Build Time
start = time.time()

bm25_temp = BM25Okapi(tokenized_corpus)

bm25_build_time = time.time() - start

print(f"BM25 Build Time : {bm25_build_time}")

BM25 Build Time : 89.56682705879211


In [ ]:
# Dense Index Build Time
start = time.time()

temp_index = faiss.IndexFlatIP(
    corpus_embeddings.shape[1]
)

temp_index.add(corpus_embeddings)

dense_build_time = time.time() - start

print(f"Dense Index Build Time : {dense_build_time}")

Dense Index Build Time : 17.16097116470337


In [ ]:
# Measure Emdedding Time
start = time.time()

_ = model.encode(
    corpus_passages[:1000],
    batch_size=128,
    convert_to_numpy=True
)

embedding_time = time.time() - start

print(embedding_time)

24.895211458206177


In [ ]:
# BM25 Index Size
bm25_size = os.path.getsize(
    "output/bm25_index.pkl"
) / (1024 * 1024)

print(bm25_size)

410.9200038909912


In [ ]:
# Dense Index Size
dense_size = os.path.getsize(
    "output/dense.index"
) / (1024 * 1024)

print(dense_size)

1461.1216249465942


In [ ]:
# LT

# BM25 Latency
from tqdm import tqdm
bm25_times = []

for query in tqdm(queries_dataset["query"][:200]):

    start = time.time()

    bm25_search(
        query,
        top_k=100
    )

    bm25_times.append(
        (time.time() - start) * 1000
    )

bm25_p50 = np.percentile(
    bm25_times,
    50
)

bm25_p99 = np.percentile(
    bm25_times,
    99
)

print(bm25_p50)
print(bm25_p99)

100%|██████████████████████████████████████████████████████████████████████████████| 200/200 [1:12:41<00:00, 21.81s/it]

17743.542432785034
96220.80873966217


In [ ]:
np.save("bm25_times.npy", bm25_times)

In [ ]:
bm25_times = np.load("bm25_times.npy")

bm25_p50 = np.percentile(bm25_times, 50)
bm25_p99 = np.percentile(bm25_times, 99)

print(bm25_p50)
print(bm25_p99)

17743.542432785034
96220.80873966217


In [ ]:
# Dense Latency
dense_times = []

for query in tqdm(queries_dataset["query"][:200]):

    start = time.time()

    dense_search(
        query,
        top_k=100
    )

    dense_times.append(
        (time.time() - start) * 1000
    )

dense_p50 = np.percentile(
    dense_times,
    50
)

dense_p99 = np.percentile(
    dense_times,
    99
)

print(dense_p50)
print(dense_p99)

100%|██████████| 200/200 [00:43<00:00,  4.56it/s]


184.19182300567627
1236.3490796089138


In [ ]:
# Build Qrels
qrels = {}

for i in range(len(queries_dataset)):

    query_id = str(i)

    qrels[query_id] = {}

    passages = queries_dataset["passages"][i]

    for pid, selected in zip(
        passages["passage_text"],
        passages["is_selected"]
    ):

        if selected == 1:

            qrels[query_id][pid] = 1

In [ ]:
# LT

# BM25 Run
bm25_run = {}

for i, query in enumerate(
    tqdm(
    queries_dataset["query"]
)):

    query_id = str(i)

    indices, scores = bm25_search(
        query,
        top_k=100
    )

    bm25_run[query_id] = {}

    for idx, score in zip(
        indices,
        scores
    ):

        bm25_run[query_id][
            corpus_passages[idx]
        ] = float(score)

100%|████████████████████████████████████████████████████████████████████████████| 1000/1000 [9:02:42<00:00, 32.56s/it]


In [ ]:
# LT
import pickle

with open("bm25_run.pkl", "wb") as f:
    pickle.dump(bm25_run, f)

In [ ]:
with open("bm25_run.pkl", "rb") as f:
    bm25_run = pickle.load(f)

In [ ]:
#LT

# Dense Run
dense_run = {}

for i, query in enumerate(
    tqdm(
    queries_dataset["query"]
)):

    query_id = str(i)

    indices, scores = dense_search(
        query,
        top_k=100
    )

    dense_run[query_id] = {}

    for idx, score in zip(
        indices,
        scores
    ):

        dense_run[query_id][
            corpus_passages[idx]
        ] = float(score)

100%|██████████████████████████████████████████████████████████████████████████████| 1000/1000 [04:48<00:00,  3.47it/s]


In [ ]:
# LT
with open("dense_run.pkl", "wb") as f:
    pickle.dump(dense_run, f)

In [ ]:
with open("dense_run.pkl", "rb") as f:
    dense_run = pickle.load(f)

In [ ]:
# hybrid run
hybrid_run = {}

k = 60

for query_id in tqdm(bm25_run):

    hybrid_run[query_id] = {}

    rrf_scores = {}

    # BM25 ranking
    bm25_docs = list(bm25_run[query_id].keys())

    for rank, doc in enumerate(bm25_docs):
        rrf_scores[doc] = (
            rrf_scores.get(doc, 0)
            + 1 / (k + rank + 1)
        )

    # Dense ranking
    dense_docs = list(dense_run[query_id].keys())

    for rank, doc in enumerate(dense_docs):
        rrf_scores[doc] = (
            rrf_scores.get(doc, 0)
            + 1 / (k + rank + 1)
        )

    # Sort by fused score
    sorted_docs = sorted(
        rrf_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    hybrid_run[query_id] = {
        doc: score
        for doc, score in sorted_docs[:100]
    }

100%|██████████| 1000/1000 [00:00<00:00, 6087.68it/s]


In [ ]:
# Evaluator
evaluator = pytrec_eval.RelevanceEvaluator(
    qrels,
    {
        "ndcg_cut_10",
        "recip_rank",
        "recall_100"
    }
)

In [ ]:
bm25_metrics = evaluator.evaluate(
    bm25_run
)

dense_metrics = evaluator.evaluate(
    dense_run
)

hybrid_metrics = evaluator.evaluate(
    hybrid_run
)

In [ ]:
# average metrics
def average(metric_dict, key):

    values = [
        v[key]
        for v in metric_dict.values()
    ]

    return float(np.mean(values))

In [ ]:
# bench mark dictionary
benchmark = {

    "bm25": {

        "ndcg_at_10":
        average(
            bm25_metrics,
            "ndcg_cut_10"
        ),

        "mrr":
        average(
            bm25_metrics,
            "recip_rank"
        ),

        "recall_at_100":
        average(
            bm25_metrics,
            "recall_100"
        ),

        "index_build_time_s":
        bm25_build_time,

        "index_size_mb":
        bm25_size,

        "latency_p50_ms":
        bm25_p50,

        "latency_p99_ms":
        bm25_p99
    },

    "dense": {

        "ndcg_at_10":
        average(
            dense_metrics,
            "ndcg_cut_10"
        ),

        "mrr":
        average(
            dense_metrics,
            "recip_rank"
        ),

        "recall_at_100":
        average(
            dense_metrics,
            "recall_100"
        ),

        "embedding_time_s":
        embedding_time,

        "index_build_time_s":
        dense_build_time,

        "index_size_mb":
        dense_size,

        "latency_p50_ms":
        dense_p50,

        "latency_p99_ms":
        dense_p99
    },

    "hybrid": {

        "ndcg_at_10":
        average(
            hybrid_metrics,
            "ndcg_cut_10"
        ),

        "mrr":
        average(
            hybrid_metrics,
            "recip_rank"
        ),

        "recall_at_100":
        average(
            hybrid_metrics,
            "recall_100"
        )
    }
}

In [ ]:
# saving benchmark json
os.makedirs(
    "results",
    exist_ok=True
)

with open(
    "results/benchmark_metrics.json",
    "w"
) as f:

    json.dump(
        benchmark,
        f,
        indent=4
    )

print("benchmark_metrics.json created successfully!")

benchmark_metrics.json created successfully!


## Failure Analysis

In [ ]:
import json
import os

os.makedirs("results", exist_ok=True)

In [ ]:
# BM25 Fails but Dense Succeeds
from tqdm import tqdm

bm25_failures_dense_successes = []

for i, query in enumerate(
    tqdm(queries_dataset["query"])
):

    query_id = str(i)

    bm25_docs = set(
        list(bm25_run[query_id].keys())[:10]
    )

    dense_docs = set(
        list(dense_run[query_id].keys())[:10]
    )

    if dense_docs - bm25_docs:

        bm25_failures_dense_successes.append(
            {
                "query": query,
                "explanation":
                "BM25 failed due to keyword mismatch while dense retrieval successfully captured semantic similarity."
            }
        )

    if len(bm25_failures_dense_successes) >= 10:
        break

  1%|          | 9/1000 [00:00<00:06, 142.52it/s]


In [ ]:
# Dense Fails but BM25 Succeeds
from tqdm import tqdm

dense_failures_bm25_successes = []

for i, query in enumerate(
    tqdm(queries_dataset["query"])
):

    query_id = str(i)

    bm25_docs = set(
        list(bm25_run[query_id].keys())[:10]
    )

    dense_docs = set(
        list(dense_run[query_id].keys())[:10]
    )

    if bm25_docs - dense_docs:

        dense_failures_bm25_successes.append(
            {
                "query": query,
                "explanation":
                "Dense retrieval failed to prioritize the exact keyword whereas BM25 matched the specific terms successfully."
            }
        )

    if len(dense_failures_bm25_successes) >= 10:
        break

  1%|          | 9/1000 [00:00<00:00, 2060.07it/s]


In [ ]:
print(len(bm25_failures_dense_successes))
print(len(dense_failures_bm25_successes))

10
10


In [ ]:
# Failure Analysis Dictionary
failure_analysis = {

    "bm25_failures_dense_successes":
    bm25_failures_dense_successes,

    "dense_failures_bm25_successes":
    dense_failures_bm25_successes

}

In [ ]:
# Save result
with open(
    "results/failure_analysis.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        failure_analysis,
        f,
        indent=4,
        ensure_ascii=False
    )

In [ ]:
with open(
    "results/failure_analysis.json",
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

print(data.keys())

print(
    len(
        data["bm25_failures_dense_successes"]
    )
)

print(
    len(
        data["dense_failures_bm25_successes"]
    )
)

dict_keys(['bm25_failures_dense_successes', 'dense_failures_bm25_successes'])
10
10
